# Gold v2 — 오프라인 agent / 앙상블 (새 추론 없음, CPU)

이미 끝난 4개 조건의 예측만 읽어서 agent·앙상블 정책을 계산합니다. **GPU 불필요 — CPU 런타임으로 실행하세요.** 1분 안에 끝납니다.

- 입력 (같은 gold_v2_final 246장 / 534문항, 4 LVLM)
  - image-only: repo `results/latest/gold_v2_baseline/prediction_review.csv`
  - normal reference: `MyDrive/lvlm_insplad_results/gold_reference_v2/normal_reference/predictions/`
  - expert only (PatchCore score): `MyDrive/lvlm_insplad_results/gold_v2_vision_expert_full/eval/predictions/`
  - normal reference + PatchCore score: `MyDrive/lvlm_insplad_results/gold_v2_reference_expert/predictions/`
- 정책
  - `A1_*`: normal reference 답이 PatchCore 점수와 강하게 충돌(정상인데 상위 10%, 이상인데 하위 10%)한 이미지만 결합 조건으로 다시 묻기 — 교체 범위 Q1 / Q1·Q2 / 전부
  - `S_q3_image_only`, `*+S`: Q3 위치는 원본만 답 사용 (사후 규칙)
  - `A5`: 점수 충돌 또는 Q1·Q2 답 모순이면 다시 묻기
  - `A2` 점수 단독, `A3` 4모델 다수결, `A4` 조건 3개 다수결
- 판정: Q1 anomaly F1 / Q2 defect macro-F1 / Q3 위치 정확도를 따로 채점
  - `all3_up_vs_normal_ref`: 세 문항 모두 normal reference보다 나음
  - `all3_at_least_best_fixed`: 세 문항 모두 같은 모델의 고정 4조건 최고값 이상, 하나는 초과 ← 진짜 기준
  - `significance.csv`: 이미지 단위 paired bootstrap 95% CI
- 충돌 기준 10/90은 AgentIAD-lite 기본값을 그대로 사용 (gold v2 결과를 보고 고르지 않음). 점수 percentile은 246장 안 상대 순위(transductive)

결과: `MyDrive/lvlm_insplad_results/gold_v2_offline_agent/` → GitHub `results/latest/gold_v2_offline_agent/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
NORMAL_REF_FOLDER = "lvlm_insplad_results/gold_reference_v2/normal_reference/predictions"
EXPERT_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full/eval/predictions"
COMBINED_FOLDER = "lvlm_insplad_results/gold_v2_reference_expert/predictions"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_offline_agent"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
EXPERT_PAIR = "patchcore__score"
CONFLICT_LOW, CONFLICT_HIGH = 10.0, 90.0   # 고정 (사후 튜닝 금지)
SCORE_ONLY_THRESHOLD = 50.0
PUBLISH = True

In [ ]:
# Drive 연결 + repo clone (pip 설치·GPU 없음)
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
NORMAL_REF_DIR = DRIVE_ROOT / NORMAL_REF_FOLDER
EXPERT_DIR = DRIVE_ROOT / EXPERT_FOLDER
COMBINED_DIR = DRIVE_ROOT / COMBINED_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
IMAGE_ONLY_CSV = REPO_DIR / "results/latest/gold_v2_baseline/prediction_review.csv"
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())

In [ ]:
# 입력 파일 확인 — 하나라도 없으면 여기서 멈춤
need = [IMAGE_ONLY_CSV]
for m in MODELS:
    need += [NORMAL_REF_DIR / f"{m}.jsonl",
             EXPERT_DIR / m / f"{EXPERT_PAIR}.jsonl",
             COMBINED_DIR / m / f"{EXPERT_PAIR}.jsonl"]
missing = [str(p) for p in need if not p.is_file()]
for p in need:
    print("있음" if p.is_file() else "없음", p)
if missing:
    raise FileNotFoundError(f"{len(missing)}개 입력 없음 — 위 '없음' 경로 확인")
if not sorted(QA_DIR.glob("master_qa_*.jsonl")):
    raise FileNotFoundError(f"master_qa_*.jsonl under {QA_DIR}")

In [ ]:
# 계산 (새 LVLM 추론 없음)
run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_offline_agent.py",
    "--qa-dir", QA_DIR,
    "--image-only-csv", IMAGE_ONLY_CSV,
    "--normal-ref-dir", NORMAL_REF_DIR,
    "--expert-dir", EXPERT_DIR,
    "--combined-dir", COMBINED_DIR,
    "--out-dir", RUN_ROOT,
    "--models", *MODELS,
    "--expert-pair", EXPERT_PAIR,
    "--conflict-low", str(CONFLICT_LOW),
    "--conflict-high", str(CONFLICT_HIGH),
    "--score-only-threshold", str(SCORE_ONLY_THRESHOLD),
])
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")

In [ ]:
# 검증: fixed_* 4조건이 GitHub에 게시된 기존 리더보드와 같은 숫자인지 (다르면 입력이 잘못된 것)
L = REPO_DIR / "results/latest"
expected = {
    "fixed_image_only": pd.read_csv(L / "gold_v2_baseline/leaderboard.csv"),
    "fixed_normal_ref": pd.read_csv(L / "gold_v2_reference/leaderboard.csv").query("condition == 'normal_reference'"),
    "fixed_expert_only": pd.read_csv(L / "gold_v2_vision_expert_full/leaderboard.csv").query("expert == 'patchcore' and condition == 'score'"),
    "fixed_combined": pd.read_csv(L / "gold_v2_reference_expert/leaderboard.csv").query("expert == 'patchcore' and condition == 'score'"),
}
cols = {"Q1_anomaly_f1": "anomaly_f1", "Q2_defect_macro_f1": "defect_macro_f1", "Q3_loc_acc": "localization_accuracy"}
bad = []
for policy, ref in expected.items():
    ref = ref.set_index("model_key")
    for _, row in lb[lb.policy == policy].iterrows():
        for ours, theirs in cols.items():
            want = float(ref.loc[row.model_key, theirs])
            if abs(float(row[ours]) - want) > 1e-6:
                bad.append((policy, row.model_key, ours, round(float(row[ours]), 4), round(want, 4)))
if bad:
    display(pd.DataFrame(bad, columns=["policy","model","metric","computed","published"]))
    raise RuntimeError("fixed 조건 숫자가 게시된 리더보드와 다릅니다 — 입력 경로/파일 확인")
print("검증 통과: fixed 4조건 × 4모델 × 3지표가 기존 리더보드와 일치")

In [ ]:
# 결과
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
print("[라우팅 비율]")
display(pd.read_csv(RUN_ROOT / "route_stats.csv").round(3))
show = ["model_key","policy","Q1_anomaly_f1","Q2_defect_macro_f1","Q3_loc_acc",
        "d_Q1_vs_normal_ref","d_Q2_vs_normal_ref","d_Q3_vs_normal_ref",
        "all3_up_vs_normal_ref","all3_at_least_best_fixed","avg_lvlm_calls_per_q"]
display(lb[show].round(3))
print("\n[세 문항 모두 고정 4조건 최고값 이상(하나는 초과)인 정책]  ← 진짜 기준")
display(lb[lb.all3_at_least_best_fixed][show].round(3))

# 유의성: 이미지 단위 paired bootstrap. p_le_0 < 0.05 이고 CI 하한 > 0 이면 '개선'으로 볼 수 있음
sig = pd.read_csv(RUN_ROOT / "significance.csv")
for q in ("Q1", "Q2", "Q3"):
    sig[f"{q}_sig"] = (sig[f"{q}_ci_low"] > 0)
print("\n[유의성] baseline 대비 (S_q3_image_only 대비 행 = agent 자체 효과)")
display(sig[["model_key","baseline","policy",
             "Q1_delta","Q1_ci_low","Q1_ci_high","Q1_p_le_0","Q1_sig",
             "Q2_delta","Q2_ci_low","Q2_ci_high","Q2_p_le_0","Q2_sig",
             "Q3_delta","Q3_ci_low","Q3_ci_high","Q3_sig"]].round(3))

In [ ]:
# GitHub publish
if PUBLISH:
    publish_files = ["leaderboard.csv", "route_stats.csv", "significance.csv", "run_metadata.json"]
    run([
        sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
        "--out-dir", RUN_ROOT,
        "--repo", REPO,
        "--branch", REPO_REF,
        "--prefix", "gold_v2_offline_agent",
        "--files", *[f for f in publish_files if (RUN_ROOT / f).is_file()],
    ], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_offline_agent/")

## 완료 후

- 검증 셀이 통과해야 결과를 믿을 수 있습니다.
- 핵심 1: `all3_at_least_best_fixed` — 같은 모델의 고정 조건(원본만 / normal ref / expert만 / 결합) 문항별 최고값 이상(하나는 초과). Q3를 원본 답으로 쓰는 정책은 Q3가 원본만과 같아서 strict `>`(`all3_beat_best_fixed`)는 구조상 불가능.
- 핵심 2: `significance.csv` — 이미지 단위 paired bootstrap 2,000회. CI 하한 > 0이면 우연이 아닌 개선. `S_q3_image_only` 대비 행이 agent 자체 효과.
- `S_q3_image_only`와 `*+S`는 결과를 보고 정한 규칙이라 논문에는 사후 분석으로 표기.